# Data Cleaning & Handling
## Dataset: loan_data_2007_2014

**Author:** Oknardo Tulung  
**LinkedIn:** https://www.linkedin.com/in/oknardo-tulung/  
**GitHub:** https://github.com/oknardo/Home_Credit_Scorecard_Model

---

## 📌 Project Overview
This notebook covers the data preparation phase of the ID/X Partners Credit Risk Prediction project.
Based on findings from the EDA phase, this notebook systematically cleans and transforms the dataset
to produce a model-ready dataframe for the modeling phase.

---

## 🎯 Objectives
- Remove irrelevant, constant, and post-origination leakage features
- Handle missing values through imputation and missing indicators
- Treat outliers in key numerical features
- Encode categorical variables for modeling
- Scale numerical features
- Split data into train and test sets

---

## 🔍 Scope
The approach includes:
- Feature dropping (all-null, constant, leakage, high cardinality)
- Missing value imputation (median for numerical, mode for categorical)
- Outlier capping at defined percentiles
- Ordinal and one-hot encoding for categorical features
- Standard scaling for numerical features
- Train/test split with stratification on target variable

---

## 🛠 Tools & Libraries
- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib
- Seaborn

# Importing Library

In [1]:
# Installation Library
!pip install seaborn scikit-learn lightgbm xgboost catboost shap imbalanced-learn 


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Install optuna
!pip install optuna


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# Hide Warning
import warnings
warnings.filterwarnings('ignore')

# Importing Library
import pandas as pd
# Setting Pandas Row Display Max
pd.set_option('display.max_rows', None)

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import math
import joblib

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import shap
# Hyperparameter Tunning
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Importing Dataset

In [4]:
# Dataset loan_data
df = r'D:\Python\Projects\Project Idxpartners Credit Risk Prediction Model\loan_data_2007_2014.csv'
df = pd.read_csv(df)

# 1. Drop Irrelevant Features

This section removes features that are not suitable for modeling based on findings from the EDA phase.

The approach includes:
- **All-null columns**: 17 columns with zero non-null values
- **Constant columns**: features with only one unique value across all records
- **Identifier columns**: `id`, `member_id`, `url`, `Unnamed: 0` carry no predictive value
- **High cardinality columns**: `emp_title`, `title`, `zip_code`, `desc` are too granular for direct use
- **Post-origination leakage features**: repayment outcome features not available at loan origination
- **Redundant columns**: highly correlated features that add no new information

In [5]:
# All-null columns
cols_all_null = [
    'annual_inc_joint', 'dti_joint', 'verification_status_joint',
    'open_acc_6m', 'open_il_6m', 'open_il_12m', 'open_il_24m',
    'mths_since_rcnt_il', 'total_bal_il', 'il_util',
    'open_rv_12m', 'open_rv_24m', 'max_bal_bc', 'all_util',
    'inq_fi', 'total_cu_tl', 'inq_last_12m'
]

# Constant columns
cols_constant = ['policy_code', 'pymnt_plan', 'application_type']

# Identifier columns
cols_identifier = ['Unnamed: 0', 'id', 'member_id', 'url']

# High cardinality columns
cols_high_cardinality = ['emp_title', 'title', 'zip_code', 'desc']

# Post-origination leakage features
cols_leakage = [
    'total_rec_prncp', 'recoveries', 'last_pymnt_amnt',
    'total_pymnt', 'total_pymnt_inv', 'collection_recovery_fee',
    'total_rec_int', 'total_rec_late_fee', 'out_prncp', 'out_prncp_inv',
    'next_pymnt_d', 'last_pymnt_d', 'last_credit_pull_d'
]

# Redundant columns (highly correlated with loan_amnt)
cols_redundant = ['funded_amnt', 'funded_amnt_inv']

# Combine all columns to drop
cols_to_drop = (
    cols_all_null + cols_constant + cols_identifier +
    cols_high_cardinality + cols_leakage + cols_redundant
)

# Drop columns
df = df.drop(columns=cols_to_drop)

print(f"Remaining features: {df.shape[1]} columns")
print(f"Dropped features: {len(cols_to_drop)} columns")

Remaining features: 32 columns
Dropped features: 43 columns


# 2. Handle Target Variable

This section defines and prepares the target variable for modeling.
Ongoing loans are removed as their final repayment outcome is unknown,
and the target is encoded as a binary variable for classification modeling.

In [6]:
# Define loan status groups
good_loan = ['Fully Paid', 'Does not meet the credit policy. Status:Fully Paid']
bad_loan = ['Charged Off', 'Default', 'Does not meet the credit policy. Status:Charged Off']

# Map loan_status to loan_group
df['loan_group'] = df['loan_status'].map(
    {status: 'Good Loan' for status in good_loan} |
    {status: 'Bad Loan' for status in bad_loan}
)

# Drop ongoing loans
df = df[df['loan_group'].isin(['Good Loan', 'Bad Loan'])].copy()

# Encode target as binary
df['is_default'] = (df['loan_group'] == 'Bad Loan').astype(int)

# Drop original loan_status and loan_group
df = df.drop(columns=['loan_status', 'loan_group'])

print(f"Dataset shape after removing Ongoing loans: {df.shape}")
print(f"\nTarget distribution:")
print(df['is_default'].value_counts())
print(f"\nDefault rate: {df['is_default'].mean() * 100:.2f}%")

Dataset shape after removing Ongoing loans: (230795, 32)

Target distribution:
is_default
0    186727
1     44068
Name: count, dtype: int64

Default rate: 19.09%


# 3. Handle Missing Values

This section handles remaining missing values based on the strategy defined in the EDA phase.

The approach includes:
- **Drop**: high missingness features with minimal predictive signal
- **Median impute**: numerical features with moderate missingness
- **Mode impute**: categorical features with low missingness
- **Missing indicator**: features where missingness itself carries predictive signal

In [7]:
# Drop high missingness features (>50%)
cols_high_missing = [
    'mths_since_last_record',
    'mths_since_last_major_derog',
    'mths_since_last_delinq'
]

df = df.drop(columns=cols_high_missing)

print(f"Dropped high missingness columns: {cols_high_missing}")

Dropped high missingness columns: ['mths_since_last_record', 'mths_since_last_major_derog', 'mths_since_last_delinq']


In [ ]:
# emp_length: missingness shows higher default rate (27%)
df['emp_length_MISSING'] = df['emp_length'].isna().astype(int)

print("Missing indicator created for emp_length")

Missing indicator created for emp_length


In [9]:
# Median impute for numerical features
num_impute_cols = ['revol_util', 'collections_12_mths_ex_med',
                   'delinq_2yrs', 'inq_last_6mths', 'open_acc',
                   'acc_now_delinq', 'pub_rec', 'total_acc',
                   'tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim',
                   'annual_inc']

for col in num_impute_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

print(f"Median imputed: {num_impute_cols}")

Median imputed: ['revol_util', 'collections_12_mths_ex_med', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'acc_now_delinq', 'pub_rec', 'total_acc', 'tot_cur_bal', 'tot_coll_amt', 'total_rev_hi_lim', 'annual_inc']


In [10]:
# Mode impute for categorical features
cat_impute_cols = ['emp_length', 'earliest_cr_line', 'last_pymnt_d', 'title']

for col in cat_impute_cols:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        df[col] = df[col].fillna(mode_val)

print(f"Mode imputed: {cat_impute_cols}")

Mode imputed: ['emp_length', 'earliest_cr_line', 'last_pymnt_d', 'title']


In [11]:
# Verify no missing values remain
missing_remaining = df.isnull().sum()
missing_remaining = missing_remaining[missing_remaining > 0]

if len(missing_remaining) == 0:
    print("No missing values remaining.")
else:
    print(f"Remaining missing values:\n{missing_remaining}")

No missing values remaining.


# 4. Handle Outliers

This section treats outliers in key numerical features based on findings from the EDA phase.

The approach includes:
- **Capping at 99th percentile**: for right-skewed financial features with extreme upper values
- **Hard capping at 100%**: for `revol_util` which has a defined valid range of 0–100%

In [12]:
# Features to cap at 99th percentile
cols_to_cap = [
    'annual_inc', 'revol_bal', 'tot_coll_amt',
    'tot_cur_bal', 'total_rev_hi_lim', 'installment'
]

for col in cols_to_cap:
    cap_value = df[col].quantile(0.99)
    df[col] = df[col].clip(upper=cap_value)
    print(f"{col}: capped at {cap_value:,.2f}")

annual_inc: capped at 235,000.00
revol_bal: capped at 80,631.22
tot_coll_amt: capped at 1,839.06
tot_cur_bal: capped at 585,250.60
total_rev_hi_lim: capped at 116,853.00
installment: capped at 1,192.64


In [13]:
# Hard cap revol_util at 100%
df['revol_util'] = df['revol_util'].clip(upper=100)

print(f"revol_util: hard capped at 100%")

revol_util: hard capped at 100%


In [14]:
# Verify outlier treatment
print("\nPost-capping statistics:")
check_cols = cols_to_cap + ['revol_util']
df[check_cols].describe().T[['min', '50%', '99%', 'max']] if '99%' in df[check_cols].describe().T.columns else df[check_cols].describe().T[['min', '50%', 'max']]


Post-capping statistics:


,min,50%,max
annual_inc,1896.00,61920.00,235000.00
revol_bal,0.00,10979.00,80631.22
tot_coll_amt,0.00,0.00,1839.06
tot_cur_bal,0.00,80282.00,585250.60
total_rev_hi_lim,0.00,22100.00,116853.00
installment,15.67,363.85,1192.64
revol_util,0.00,56.50,100.00


# 5. Feature Engineering

This section derives new features from existing columns to improve model predictive power.

The approach includes:
- **Credit age**: derived from `earliest_cr_line` as the number of years since first credit line
- **Loan issue year**: extracted from `issue_d` to capture loan vintage effects
- **Employment length**: converted from string to ordinal numeric values
- **Dropping original date columns** after feature extraction

In [15]:
# Convert issue_d to loan issue year
df['issue_year'] = pd.to_datetime(df['issue_d'], format='%b-%y').dt.year

# Convert earliest_cr_line to credit age in years
df['credit_age_years'] = (
    pd.to_datetime(df['issue_d'], format='%b-%y').dt.year -
    pd.to_datetime(df['earliest_cr_line'], format='%b-%y').dt.year
)

# Drop original date columns
df = df.drop(columns=['issue_d', 'earliest_cr_line'])

print("Created: issue_year, credit_age_years")
print(f"Dropped: issue_d, earliest_cr_line")

Created: issue_year, credit_age_years
Dropped: issue_d, earliest_cr_line


In [16]:
# Convert emp_length to ordinal numeric
emp_length_map = {
    '< 1 year': 0, '1 year': 1, '2 years': 2, '3 years': 3,
    '4 years': 4, '5 years': 5, '6 years': 6, '7 years': 7,
    '8 years': 8, '9 years': 9, '10+ years': 10
}

df['emp_length'] = df['emp_length'].map(emp_length_map)

print("emp_length converted to ordinal numeric")
print(df['emp_length'].value_counts().sort_index())

emp_length converted to ordinal numeric
emp_length
0     19319
1     15575
2     21647
3     18761
4     14940
5     16821
6     13845
7     13034
8     10698
9      8639
10    77516
Name: count, dtype: int64


In [17]:
# Verify new features
print(f"\nDataset shape: {df.shape}")
df[['issue_year', 'credit_age_years', 'emp_length']].describe().T


Dataset shape: (230795, 30)


,count,mean,std,min,25%,50%,75%,max
issue_year,230795.0,2012.593033,1.377827,2007.0,2012.0,2013.0,2014.0,2014.0
credit_age_years,230795.0,14.898057,7.526389,-60.0,10.0,14.0,18.0,45.0
emp_length,230795.0,5.943890,3.635128,0.0,3.0,6.0,10.0,10.0


In [18]:
# Investigate negative credit_age_years
print(f"Negative credit_age_years count: {(df['credit_age_years'] < 0).sum()}")
print(f"\nSample of negative values:")
print(df[df['credit_age_years'] < 0][['credit_age_years']].value_counts().head(10))

Negative credit_age_years count: 498

Sample of negative values:
credit_age_years
-54                 109
-53                  72
-55                  58
-52                  50
-50                  37
-51                  33
-56                  30
-57                  19
-49                  16
-48                  15
Name: count, dtype: int64


In [19]:
# Clip credit_age_years to 0 minimum (negative values are parsing artifacts)
df['credit_age_years'] = df['credit_age_years'].clip(lower=0)

print(f"Negative credit_age_years remaining: {(df['credit_age_years'] < 0).sum()}")
print(f"\nUpdated stats:")
print(df['credit_age_years'].describe())

Negative credit_age_years remaining: 0

Updated stats:
count    230795.000000
mean         15.011153
std           6.878170
min           0.000000
25%          10.000000
50%          14.000000
75%          18.000000
max          45.000000
Name: credit_age_years, dtype: float64


# 6. Encoding

This section encodes categorical features into numerical representations suitable for modeling.

The approach includes:
- **Ordinal encoding**: for features with natural order (`grade`, `sub_grade`)
- **One-hot encoding**: for nominal categorical features with no inherent order
- **Dropping original columns** after encoding

In [20]:
# Ordinal encoding for grade
grade_map = {'A': 1, 'B': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7}
df['grade'] = df['grade'].map(grade_map)

print("grade encoded ordinally:")
print(df['grade'].value_counts().sort_index())

grade encoded ordinally:
grade
1    39110
2    70622
3    58626
4    36740
5    16945
6     6896
7     1856
Name: count, dtype: int64


In [21]:
# Ordinal encoding for sub_grade
sub_grades = sorted(df['sub_grade'].unique())
sub_grade_map = {sg: i+1 for i, sg in enumerate(sub_grades)}
df['sub_grade'] = df['sub_grade'].map(sub_grade_map)

print(f"\nsub_grade encoded ordinally (1 to {len(sub_grade_map)})")


sub_grade encoded ordinally (1 to 35)


In [22]:
# One-hot encoding for nominal categorical features
cat_ohe_cols = [
    'term', 'home_ownership', 'verification_status',
    'purpose', 'initial_list_status', 'addr_state'
]

df = pd.get_dummies(df, columns=cat_ohe_cols, drop_first=True)

print(f"One-hot encoded: {cat_ohe_cols}")
print(f"\nDataset shape after encoding: {df.shape}")

One-hot encoded: ['term', 'home_ownership', 'verification_status', 'purpose', 'initial_list_status', 'addr_state']

Dataset shape after encoding: (230795, 95)


# 7. Train/Test Split

In [26]:
# Separate features and target
X = df.drop(columns=['is_default'])
y = df['is_default']

# Split before scaling to prevent data leakage
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train set: {X_train.shape}")
print(f"Test set : {X_test.shape}")
print(f"\nTarget distribution in train set:")
print(y_train.value_counts())
print(f"\nTarget distribution in test set:")
print(y_test.value_counts())

Train set: (184636, 94)
Test set : (46159, 94)

Target distribution in train set:
is_default
0    149382
1     35254
Name: count, dtype: int64

Target distribution in test set:
is_default
0    37345
1     8814
Name: count, dtype: int64


# 8. Scaling

This section applies standard scaling to numerical features to normalize their range.
Scaling ensures that features with large value ranges do not dominate the model,
particularly for Logistic Regression which is sensitive to feature scale.

The approach includes:
- **StandardScaler**: transforms features to have mean = 0 and standard deviation = 1
- Scaling is applied only to numerical features, excluding the target variable and binary columns

In [27]:
# Numerical features to scale
cols_to_scale = [
    'loan_amnt', 'int_rate', 'installment', 'annual_inc', 'dti',
    'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec',
    'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med',
    'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim',
    'grade', 'sub_grade', 'emp_length',
    'issue_year', 'credit_age_years', 'emp_length_MISSING'
]

# Fit on train, transform both train and test
scaler = StandardScaler()
X_train[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

print("Scaling applied successfully.")
print(f"\nSample scaled values (train):")
X_train[cols_to_scale].describe().T[['mean', 'std', 'min', 'max']].head(5)

Scaling applied successfully.

Sample scaled values (train):


,mean,std,min,max
loan_amnt,4.233178e-17,1.000003,-1.604285,2.680704
int_rate,3.330356e-16,1.000003,-1.915173,2.809596
installment,-2.115050e-16,1.000003,-1.650615,3.216746
annual_inc,1.493157e-17,1.000003,-1.788043,4.259733
dti,7.488876e-17,1.000003,-2.132666,3.083931


# Save train and test sets to CSV

In [28]:
# Save train and test sets to CSV
train_path = r'D:\Python\Projects\Project Idxpartners Credit Risk Prediction Model\Loan_Credit_Risk_Prediction\train_set.csv'
test_path = r'D:\Python\Projects\Project Idxpartners Credit Risk Prediction Model\Loan_Credit_Risk_Prediction\test_set.csv'

X_train_save = X_train.copy()
X_train_save['is_default'] = y_train.values

X_test_save = X_test.copy()
X_test_save['is_default'] = y_test.values

X_train_save.to_csv(train_path, index=False)
X_test_save.to_csv(test_path, index=False)

print(f"Train set saved: {X_train_save.shape}")
print(f"Test set saved : {X_test_save.shape}")

Train set saved: (184636, 95)
Test set saved : (46159, 95)


# Summary

This notebook successfully prepared the dataset for the modeling phase.
Below is a summary of all cleaning and transformation steps applied.

| Step | Section | Result |
|------|---------|--------|
| 1 | Drop Irrelevant Features | Reduced from 75 to 32 columns (43 columns dropped) |
| 2 | Handle Target Variable | Reduced from 466,285 to 230,795 rows (Ongoing loans removed) |
| 3 | Handle Missing Values | 0 missing values remaining after imputation |
| 4 | Handle Outliers | Capped at 99th percentile, `revol_util` hard capped at 100% |
| 5 | Feature Engineering | Added `issue_year`, `credit_age_years`, `emp_length` as ordinal |
| 6 | Encoding | Ordinal encoding for `grade`, `sub_grade`; OHE for 6 categorical features |
| 7 | Train/Test Split | Train: 184,636 rows / Test: 46,159 rows (80/20, stratified) |
| 8 | Scaling | StandardScaler applied, fit on train set only to prevent data leakage |

**Final dataset shape:**
- Train set: 184,636 rows x 95 columns
- Test set: 46,159 rows x 95 columns
- Target: `is_default` (0 = Good Loan, 1 = Bad Loan)
- Default rate: 19.09%